# MiDas

## MiDas Full Res

In [16]:
import torch
import cv2
import time
import numpy as np
from midas.model_loader import load_model, default_models

first_execution = True

def process(device, model, model_type, image, input_size, target_size, optimize):
    global first_execution
    sample = torch.from_numpy(image).to(device).unsqueeze(0)

    if optimize and device == torch.device("cuda"):
        if first_execution:
            print("Optimization to half-floats activated.")
        sample = sample.to(memory_format=torch.channels_last)
        sample = sample.half()

    if first_execution:
        height, width = sample.shape[2:]
        print(f"Input resized to {width}x{height} before encoder")
        first_execution = False

    prediction = model.forward(sample)
    prediction = (
        torch.nn.functional.interpolate(
            prediction.unsqueeze(1),
            size=target_size[::-1],
            mode="bicubic",
            align_corners=False,
        )
        .squeeze()
        .cpu()
        .numpy()
    )
    return prediction

def normalize_depth(depth):
    depth_min = depth.min()
    depth_max = depth.max()
    normalized = 255 * (depth - depth_min) / (depth_max - depth_min)
    normalized = np.clip(normalized, 0, 255).astype(np.uint8)
    return cv2.applyColorMap(normalized, cv2.COLORMAP_INFERNO)

def main():
    # Configuration
    model_type = "dpt_next_vit_large_384"
    video_path = r"C:\Users\Pongpiched Rakshit\Documents\Adobe\Premiere Pro\25.0\videoplayback_1_1.mp4"
    optimize = False

    # Device
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"Device: {device}")

    # Load model
    model_weights = default_models[model_type]
    model, transform, net_w, net_h = load_model(device, model_weights, model_type, optimize, height=None, square=False)

    # Video
    cap = cv2.VideoCapture(video_path)
    if not cap.isOpened():
        print(f"Error opening video: {video_path}")
        return

    fps = 1
    time_start = time.time()

    while True:
        ret, frame = cap.read()
        if not ret:
            break

        # Original window
        cv2.imshow("Original Video", frame)

        # Depth estimation
        rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        image = transform({"image": rgb / 255.0})["image"]

        with torch.no_grad():
            prediction = process(device, model, model_type, image, (net_w, net_h), rgb.shape[1::-1], optimize)

        depth_vis = normalize_depth(prediction)
        cv2.imshow("Depth Map", depth_vis)

        # FPS display
        now = time.time()
        fps = 0.9 * fps + 0.1 * (1 / (now - time_start))
        time_start = now
        print(f"\rFPS: {fps:.2f}", end="")

        if cv2.waitKey(1) == 27:  # ESC
            break

    cap.release()
    cv2.destroyAllWindows()
    print("\nDone.")

if __name__ == "__main__":
    main()


Device: cuda
initialize_weights...
Model loaded, number of parameters = 72M
Input resized to 672x384 before encoder
FPS: 9.81
Done.


## MiDas 9:16

In [19]:
import torch
import cv2
import time
import numpy as np
from midas.model_loader import load_model, default_models

first_execution = True

def process(device, model, model_type, image, input_size, target_size, optimize):
    global first_execution
    sample = torch.from_numpy(image).to(device).unsqueeze(0)

    if optimize and device == torch.device("cuda"):
        if first_execution:
            print("Optimization to half-floats activated.")
        sample = sample.to(memory_format=torch.channels_last)
        sample = sample.half()

    if first_execution:
        height, width = sample.shape[2:]
        print(f"Input resized to {width}x{height} before encoder")
        first_execution = False

    prediction = model.forward(sample)
    prediction = (
        torch.nn.functional.interpolate(
            prediction.unsqueeze(1),
            size=target_size[::-1],
            mode="bicubic",
            align_corners=False,
        )
        .squeeze()
        .cpu()
        .numpy()
    )
    return prediction

def normalize_depth(depth):
    depth_min = depth.min()
    depth_max = depth.max()
    normalized = 255 * (depth - depth_min) / (depth_max - depth_min)
    normalized = np.clip(normalized, 0, 255).astype(np.uint8)
    return cv2.applyColorMap(normalized, cv2.COLORMAP_INFERNO)

def crop_center_9_16(frame):
    h, w = frame.shape[:2]
    # Desired output aspect ratio = 9:16 (w:h)
    desired_w = int(h * 9 / 16)
    start_x = (w - desired_w) // 2
    return frame[:, start_x:start_x+desired_w]

def main():
    # Configuration
    model_type = "dpt_next_vit_large_384"
    video_path = r"C:\Users\Pongpiched Rakshit\Documents\Adobe\Premiere Pro\25.0\videoplayback_1_1.mp4"
    optimize = False
    resize_inference = 384  # Reduce resolution for speed

    # Device
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"Device: {device}")

    # Load model
    model_weights = default_models[model_type]
    model, transform, net_w, net_h = load_model(
        device, model_weights, model_type, optimize,
        height=resize_inference, square=True
    )

    # Video
    cap = cv2.VideoCapture(video_path)
    if not cap.isOpened():
        print(f"Error opening video: {video_path}")
        return

    fps = 1
    time_start = time.time()

    while True:
        ret, frame = cap.read()
        if not ret:
            break

        # Crop to center 9:16
        cropped = crop_center_9_16(frame)

        # Show cropped original
        cv2.imshow("Original Cropped (9:16)", cropped)

        # Depth estimation
        rgb = cv2.cvtColor(cropped, cv2.COLOR_BGR2RGB)
        image = transform({"image": rgb / 255.0})["image"]

        with torch.no_grad():
            prediction = process(device, model, model_type, image, (net_w, net_h), rgb.shape[1::-1], optimize)
            

            h, w = prediction.shape

            # Reference: bottom-center pixel is 1.0 m
            ref_distance_m = 1.0
            ref_value = prediction[h-1, w//2]  # MiDaS raw at reference

            # Scale constant k = z_ref * d_ref so that z = k / d
            k = ref_distance_m * ref_value
            eps = 1e-8
            prediction_meters = k / (prediction + eps)

            # Quick sanity check prints
            print(f"Bottom center raw: {ref_value:.4f}")
            print(f"Bottom center (should be 1.0 m): {prediction_meters[h-1, w//2]:.2f} m")
            print(f"Center raw: {prediction[h//2, w//2]:.4f}")
            print(f"Center (meters): {prediction_meters[h//2, w//2]:.2f} m")



        depth_vis = normalize_depth(prediction)
        cv2.imshow("Depth Map", depth_vis)

        # FPS display
        now = time.time()
        fps = 0.9 * fps + 0.1 * (1 / (now - time_start))
        time_start = now
        print(f"\n\rFPS: {fps:.2f}", end="")

        if cv2.waitKey(1) == 27:  # ESC
            break

    cap.release()
    cv2.destroyAllWindows()
    print("\nDone.")

if __name__ == "__main__":
    main()

Device: cuda
initialize_weights...
Model loaded, number of parameters = 72M
Input resized to 384x384 before encoder
Bottom center raw: 2714.1265
Bottom center (should be 1.0 m): 1.00 m
Center raw: 528.9526
Center (meters): 5.13 m

FPS: 1.71Bottom center raw: 2722.3003
Bottom center (should be 1.0 m): 1.00 m
Center raw: 538.3671
Center (meters): 5.06 m

FPS: 3.01Bottom center raw: 2705.9482
Bottom center (should be 1.0 m): 1.00 m
Center raw: 537.7951
Center (meters): 5.03 m

FPS: 4.30Bottom center raw: 2700.7354
Bottom center (should be 1.0 m): 1.00 m
Center raw: 530.8909
Center (meters): 5.09 m

FPS: 5.50Bottom center raw: 2740.4114
Bottom center (should be 1.0 m): 1.00 m
Center raw: 530.2941
Center (meters): 5.17 m

FPS: 6.70Bottom center raw: 2716.0249
Bottom center (should be 1.0 m): 1.00 m
Center raw: 528.4196
Center (meters): 5.14 m

FPS: 7.73Bottom center raw: 2714.1831
Bottom center (should be 1.0 m): 1.00 m
Center raw: 505.9055
Center (meters): 5.37 m

FPS: 8.59Bottom center ra

# Object Detect + Rail Segment

In [5]:
from ultralytics import YOLO
import cv2
import time
import numpy as np

# Load models
detect_model = YOLO('yolo11n.pt')
segment_model = YOLO(r"C:\Users\Pongpiched Rakshit\Downloads\best (4).pt").to('cuda')

# Video input
#video_path = r"C:\Users\Pongpiched Rakshit\Downloads\Drivers view Thailand, Wongwian Yai to Maha Chai, Feb 2025 - YouTube - Google Chrome 2025-06-21 14-12-05.mp4"
video_path = r"C:\Users\Pongpiched Rakshit\Documents\Adobe\Premiere Pro\25.0\videoplayback_1_1.mp4"
cap = cv2.VideoCapture(video_path)

frame_width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
frame_height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
fps_input = cap.get(cv2.CAP_PROP_FPS)
center_x1 = int(frame_width * 0.4)
center_x2 = int(frame_width * 0.6)
prev_time = time.time()

# Output video writer
output_path = r"C:\Users\Pongpiched Rakshit\Downloads\output_video2.mp4"
fourcc = cv2.VideoWriter_fourcc(*'mp4v')
out_writer = cv2.VideoWriter(output_path, fourcc, fps_input, (frame_width, frame_height))

window_name = "YOLOv8 Combined Danger Detection"
cv2.namedWindow(window_name, cv2.WND_PROP_FULLSCREEN)
cv2.setWindowProperty(window_name, cv2.WND_PROP_FULLSCREEN, cv2.WINDOW_FULLSCREEN)

while cap.isOpened():
    ret, frame = cap.read()
    if not ret:
        break

    annotated_frame = frame.copy()

    # Segmentation
    seg_results = segment_model.predict(source=frame, device=0, conf=0.5)
    danger_mask = np.zeros((frame_height, frame_width), dtype=np.uint8)

    if seg_results[0].masks is not None:
        best_overlap = 0
        best_polygon_mask = np.zeros((frame_height, frame_width), dtype=np.uint8)

        for polygon in seg_results[0].masks.xy:
            polygon = polygon.astype(int)
            poly_mask = np.zeros((frame_height, frame_width), dtype=np.uint8)
            cv2.fillPoly(poly_mask, [polygon], 1)

            overlap_ratio = np.mean(poly_mask[:, center_x1:center_x2])
            if overlap_ratio > best_overlap:
                best_overlap = overlap_ratio
                best_polygon_mask = poly_mask

        if best_overlap > 0.05:
            rows = frame_height
            for y in range(rows - 1, -1, -5):
                max_width = 500
                min_width = -500
                dilation_width = int(min_width + (max_width - min_width) * (y / rows))
                kernel_size = dilation_width if dilation_width % 2 == 1 else dilation_width + 1

                if kernel_size > 1:
                    kernel = cv2.getStructuringElement(cv2.MORPH_RECT, (kernel_size, 1))
                    slice_mask = best_polygon_mask[y:y+5, :]
                    dilated_slice = cv2.dilate(slice_mask, kernel, iterations=1)
                    danger_mask[y:y+5, :] = dilated_slice

            mask_3c = np.repeat(danger_mask[:, :, np.newaxis], 3, axis=2)
            blue_overlay = np.zeros_like(annotated_frame)
            blue_overlay[:, :, 0] = 255
            blended = cv2.addWeighted(blue_overlay, 0.5, annotated_frame, 0.5, 0)
            annotated_frame = np.where(mask_3c == 1, blended, annotated_frame)

    # Object detection
    det_results = detect_model.predict(source=frame, device=0, conf=0.2)
    boxes = det_results[0].boxes.xyxy.cpu().numpy().astype(int)

    for box in boxes:
        x1, y1, x2, y2 = box
        x1, y1 = max(0, x1), max(0, y1)
        x2, y2 = min(frame_width - 1, x2), min(frame_height - 1, y2)

        box_area = (x2 - x1) * (y2 - y1)
        intersect_area = 0

        if box_area > 0:
            obj_mask = danger_mask[y1:y2, x1:x2]
            intersect_area = np.sum(obj_mask > 0)

        overlap_ratio = intersect_area / box_area if box_area > 0 else 0
        color = (0, 0, 255) if overlap_ratio > 0.2 else (0, 255, 0)

        cv2.rectangle(annotated_frame, (x1, y1), (x2, y2), color, 2)

    # FPS display
    curr_time = time.time()
    fps = 1 / (curr_time - prev_time)
    prev_time = curr_time
    cv2.putText(annotated_frame, f'FPS: {fps:.2f}', (10, 30),
                cv2.FONT_HERSHEY_SIMPLEX, 1, (0, 255, 0), 2)

    # Show and save
    cv2.imshow(window_name, annotated_frame)
    out_writer.write(annotated_frame)  # Save frame to output video

    if cv2.waitKey(1) & 0xFF == ord('q'):
        break

cap.release()
out_writer.release()
cv2.destroyAllWindows()


100%|██████████| 5.35M/5.35M [00:00<00:00, 6.48MB/s]



0: 384x640 1 railway, 64.2ms
Speed: 3.5ms preprocess, 64.2ms inference, 131.3ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 3 persons, 2 motorcycles, 8 umbrellas, 19.0ms
Speed: 1.7ms preprocess, 19.0ms inference, 2.8ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 1 railway, 9.8ms
Speed: 1.6ms preprocess, 9.8ms inference, 1.4ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 4 persons, 1 motorcycle, 7 umbrellas, 6.1ms
Speed: 1.7ms preprocess, 6.1ms inference, 0.9ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 1 railway, 10.9ms
Speed: 1.5ms preprocess, 10.9ms inference, 2.8ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 4 persons, 1 motorcycle, 7 umbrellas, 6.3ms
Speed: 1.6ms preprocess, 6.3ms inference, 1.0ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 1 railway, 9.5ms
Speed: 1.8ms preprocess, 9.5ms inference, 2.4ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 3 persons, 8 umbrellas,

# MiDas + Object Detect + Rail Segment

In [2]:
import cv2
import time
import numpy as np
import torch
from ultralytics import YOLO

# ---- MiDaS imports (from your repo) ----
from midas.model_loader import load_model, default_models

# ----------------------------
# CONFIGURATION
# ----------------------------
VIDEO_PATH = r"C:\Users\Pongpiched Rakshit\Documents\Adobe\Premiere Pro\25.0\videoplayback_1.mp4"
OUTPUT_PATH = r"C:\Users\Pongpiched Rakshit\Downloads\output_with_depth.mp4"
SEGMENT_MODEL = r"C:\Users\Pongpiched Rakshit\Downloads\best (4).pt"

SAVE_OUTPUT = False

DETECT_MODEL = "yolo11n.pt"

MIDAS_MODEL_TYPE = "dpt_next_vit_large_384"
MIDAS_OPTIMIZE = False
MIDAS_HEIGHT = 384
ASSUME_BOTTOM_CENTER_M = 1.0 # Assume bottom-center pixel is this distance in meters

DANGER_OVERLAP_THRESH = 0.20 # Bounding box and ROI overlap ratio threshold to consider "danger"

# ----------------------------
# Helpers for MiDaS
# ----------------------------
_first_execution = True

def process_midas(device, model, model_type, image_np, target_size, optimize):
    """Forward MiDaS and resize prediction to target_size (w,h)."""
    global _first_execution
    sample = torch.from_numpy(image_np).to(device).unsqueeze(0)

    if optimize and device.type == "cuda":
        if _first_execution:
            print("MiDaS: half precision optimization ON")
        sample = sample.to(memory_format=torch.channels_last).half()

    if _first_execution:
        h, w = sample.shape[2:]
        print(f"MiDaS encoder input: {w}x{h}")
        _first_execution = False

    pred = model.forward(sample)
    pred = (
        torch.nn.functional.interpolate(
            pred.unsqueeze(1),
            size=target_size[::-1],
            mode="bicubic",
            align_corners=False,
        )
        .squeeze()
        .detach()
        .float()
        .cpu()
        .numpy()
    )
    return pred

def crop_center_9_16(frame_bgr):
    """Crop to center 9:16 window (portrait) using full frame height."""
    h, w = frame_bgr.shape[:2]
    crop_w = int(h * 9 / 16)
    start_x = max((w - crop_w) // 2, 0)
    end_x = min(start_x + crop_w, w)
    return frame_bgr[:, start_x:end_x], start_x, end_x

# ----------------------------
# Init models / devices
# ----------------------------
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")

midas_weights = default_models[MIDAS_MODEL_TYPE]
midas_model, midas_transform, net_w, net_h = load_model(
    device,
    midas_weights,
    MIDAS_MODEL_TYPE,
    MIDAS_OPTIMIZE,
    height=MIDAS_HEIGHT,
    square=True,
)
if MIDAS_OPTIMIZE and device.type == "cuda":
    midas_model = midas_model.half()

torch.backends.cudnn.enabled = True
torch.backends.cudnn.benchmark = True

detect_model = YOLO(DETECT_MODEL).to(0 if device.type == "cuda" else "cpu")
segment_model = YOLO(SEGMENT_MODEL).to(0 if device.type == "cuda" else "cpu")

# ----------------------------
# Video loop
# ----------------------------
cap = cv2.VideoCapture(VIDEO_PATH)
if not cap.isOpened():
    raise RuntimeError(f"Could not open video: {VIDEO_PATH}")

frame_w = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
frame_h = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
fps_in = cap.get(cv2.CAP_PROP_FPS) or 30

# --- Init writer only if enabled ---
out_writer = None
if SAVE_OUTPUT:
    fourcc = cv2.VideoWriter_fourcc(*'mp4v')
    out_writer = cv2.VideoWriter(OUTPUT_PATH, fourcc, fps_in, (frame_w, frame_h))

win = "YOLO + Rails + Depth"
cv2.namedWindow(win, cv2.WND_PROP_FULLSCREEN)
cv2.setWindowProperty(win, cv2.WND_PROP_FULLSCREEN, cv2.WINDOW_FULLSCREEN)

prev_time = time.time()

while True:
    ret, frame = cap.read()
    if not ret:
        break

    annotated = frame.copy()

    # ---------- Segmentation (rails) ----------
    seg_results = segment_model.predict(source=frame, device=0, conf=0.5, verbose=False)
    rails_mask = np.zeros((frame_h, frame_w), dtype=np.uint8)

    if seg_results and seg_results[0].masks is not None:
        center_x1 = int(frame_w * 0.4)
        center_x2 = int(frame_w * 0.6)
        best_overlap = 0.0
        best_polygon_mask = np.zeros((frame_h, frame_w), dtype=np.uint8)

        for polygon in seg_results[0].masks.xy:
            polygon = polygon.astype(int)
            poly_mask = np.zeros((frame_h, frame_w), dtype=np.uint8)
            cv2.fillPoly(poly_mask, [polygon], 1)
            overlap_ratio = np.mean(poly_mask[:, center_x1:center_x2])
            if overlap_ratio > best_overlap:
                best_overlap = overlap_ratio
                best_polygon_mask = poly_mask

        if best_overlap > 0.05:
            rows = frame_h
            for y in range(rows - 1, -1, -5):
                max_width = 500
                min_width = -500
                dilation_width = int(min_width + (max_width - min_width) * (y / rows))
                ksize = dilation_width if dilation_width % 2 == 1 else dilation_width + 1
                if ksize > 1:
                    kernel = cv2.getStructuringElement(cv2.MORPH_RECT, (ksize, 1))
                    slice_mask = best_polygon_mask[y:y+5, :]
                    dilated_slice = cv2.dilate(slice_mask, kernel, iterations=1)
                    rails_mask[y:y+5, :] = dilated_slice

    # visualize rails (blue overlay)
    mask_3c = rails_mask[..., None]
    if mask_3c.any():
        blue = np.zeros_like(annotated); blue[..., 0] = 255
        blended = cv2.addWeighted(blue, 0.5, annotated, 0.5, 0)
        annotated = np.where(mask_3c == 1, blended, annotated)

    # ---------- Detection ----------
    det_results = detect_model.predict(source=frame, device=0, conf=0.2, verbose=False)
    if not det_results:
        cv2.imshow(win, annotated)
        if SAVE_OUTPUT and out_writer is not None:
            out_writer.write(annotated)
        if cv2.waitKey(1) & 0xFF == ord('q'):
            break
        continue

    boxes = det_results[0].boxes.xyxy.cpu().numpy().astype(int)

    # ---------- Depth (MiDaS on 9:16 center crop) ----------
    cropped, xL, xR = crop_center_9_16(frame)
    rgb = cv2.cvtColor(cropped, cv2.COLOR_BGR2RGB)
    image_in = midas_transform({"image": rgb / 255.0})["image"]

    with torch.no_grad():
        pred_raw = process_midas(
            device, midas_model, MIDAS_MODEL_TYPE,
            image_in, target_size=rgb.shape[1::-1], optimize=MIDAS_OPTIMIZE
        )

    Hc, Wc = pred_raw.shape
    ref_val = pred_raw[Hc - 1, Wc // 2]
    eps = 1e-8
    scale_k = ASSUME_BOTTOM_CENTER_M * max(ref_val, eps)
    depth_meters = scale_k / (pred_raw + eps)

    # ---------- Annotate detections ----------
    for box in boxes:
        x1, y1, x2, y2 = box
        x1, y1 = max(0, x1), max(0, y1)
        x2, y2 = min(frame_w - 1, x2), min(frame_h - 1, y2)

        box_area = (x2 - x1) * (y2 - y1)
        overlap = 0.0
        if box_area > 0:
            overlap = np.sum(rails_mask[y1:y2, x1:x2] > 0) / box_area

        danger = overlap > DANGER_OVERLAP_THRESH
        color = (0, 0, 255) if danger else (0, 255, 0)

        cv2.rectangle(annotated, (x1, y1), (x2, y2), color, 2)

        depth_text = "N/A"
        if danger:
            xc1 = max(x1, xL)
            xc2 = min(x2, xR)
            if xc2 > xc1:
                lx1 = xc1 - xL
                lx2 = xc2 - xL
                ly1 = y1
                ly2 = y2
                lx1 = max(0, min(Wc - 1, lx1))
                lx2 = max(0, min(Wc, lx2))
                ly1 = max(0, min(Hc - 1, ly1))
                ly2 = max(0, min(Hc, ly2))

                if lx2 > lx1 and ly2 > ly1:
                    patch = depth_meters[ly1:ly2, lx1:lx2]
                    if patch.size > 0:
                        closest = float(np.min(patch))
                        depth_text = f"{closest:.2f} m"

        if danger:
            cv2.putText(
                annotated, depth_text, (x1, max(0, y1 - 8)),
                cv2.FONT_HERSHEY_SIMPLEX, 0.6, color, 2, cv2.LINE_AA
            )

    # ---------- FPS ----------
    now = time.time()
    fps = 1.0 / max(now - prev_time, 1e-6)
    prev_time = now
    cv2.putText(annotated, f"FPS: {fps:.2f}", (10, 30),
                cv2.FONT_HERSHEY_SIMPLEX, 1, (0, 255, 0), 2, cv2.LINE_AA)

    cv2.imshow(win, annotated)

    # --- Write only if enabled ---
    if SAVE_OUTPUT and out_writer is not None:
        out_writer.write(annotated)

    if cv2.waitKey(1) & 0xFF == ord('q'):
        break

cap.release()
if SAVE_OUTPUT and out_writer is not None:
    out_writer.release()
cv2.destroyAllWindows()


Device: cuda
initialize_weights...
Model loaded, number of parameters = 72M
MiDaS encoder input: 384x384


# MiDas + Object Detect + Rail Segment + LRF

In [5]:
import cv2
import time
import numpy as np
import torch
import serial
import threading
from ultralytics import YOLO
from midas.model_loader import load_model, default_models

# ----------------------------
# CONFIGURATION
# ----------------------------
VIDEO_PATH = r"C:\Users\Pongpiched Rakshit\Documents\Adobe\Premiere Pro\25.0\videoplayback_1.mp4"
OUTPUT_PATH = r"C:\Users\Pongpiched Rakshit\Downloads\output_with_depth.mp4"
SEGMENT_MODEL = r"C:\Users\Pongpiched Rakshit\Downloads\best (4).pt"

SAVE_OUTPUT = False                 # <- toggle saving
OUTPUT_CODEC = 'mp4v'

DETECT_MODEL = "yolo11n.pt"

MIDAS_MODEL_TYPE = "dpt_next_vit_large_384"
MIDAS_OPTIMIZE = False
MIDAS_HEIGHT = 384
ASSUME_BOTTOM_CENTER_M = 1.0        # bottom-center pixel equals this many meters

DANGER_OVERLAP_THRESH = 0.20        # rails overlap → red bbox

# LRF serial ports
BIG_LRF_PORT = "COM8"
SMALL_LRF_PORT = "COM9"
LRF_BAUDRATE = 115200

# ----------------------------
# Shared LRF values
# ----------------------------
big_lrf_value = "Error"    # will update if sensor is connected
small_lrf_value = "Error"

# ----------------------------
# MiDaS helper
# ----------------------------
_first_execution = True
def process_midas(device, model, model_type, image_np, target_size, optimize):
    """Forward MiDaS and resize prediction to target_size (w,h)."""
    global _first_execution
    sample = torch.from_numpy(image_np).to(device).unsqueeze(0)

    if optimize and device.type == "cuda":
        if _first_execution:
            print("MiDaS: half precision optimization ON")
        sample = sample.to(memory_format=torch.channels_last).half()

    if _first_execution:
        h, w = sample.shape[2:]
        print(f"MiDaS encoder input: {w}x{h}")
        _first_execution = False

    pred = model.forward(sample)
    pred = (
        torch.nn.functional.interpolate(
            pred.unsqueeze(1),
            size=target_size[::-1],
            mode="bicubic",
            align_corners=False,
        )
        .squeeze()
        .detach()
        .float()
        .cpu()
        .numpy()
    )
    return pred  # (H,W), inverse-depth-ish scale

def crop_center_9_16(frame_bgr):
    """Crop to center 9:16 strip using full frame height."""
    h, w = frame_bgr.shape[:2]
    crop_w = int(h * 9 / 16)
    start_x = max((w - crop_w) // 2, 0)
    end_x = min(start_x + crop_w, w)
    return frame_bgr[:, start_x:end_x], start_x, end_x

# ----------------------------
# LRF Readers
# ----------------------------
def lrf_reader(port, which):
    global big_lrf_value, small_lrf_value
    start_cmd = bytes([0xFA,0x01,0xFF,0x04,0x01,0x00,0x00,0x00,0x00,0xFF])
    stop_cmd  = bytes([0xFA,0x01,0xFF,0x04,0x00,0x00,0x00,0x00,0x00,0xFE])
    try:
        ser = serial.Serial(port, LRF_BAUDRATE, timeout=1)
        time.sleep(1)
        ser.write(start_cmd)
        buffer = bytearray()
        while True:
            if ser.in_waiting:
                buffer += ser.read(ser.in_waiting)
                while len(buffer) >= 9:
                    if buffer[0] == 0xFB and buffer[1] == 0x03:
                        frame = buffer[:9]
                        buffer = buffer[9:]
                        valid_flag = int.from_bytes(frame[4:6], 'little')
                        distance_dm = int.from_bytes(frame[6:8], 'little')
                        txt = f"{distance_dm/10:.1f} m" if valid_flag == 1 else "Invalid"
                        if which == "BIG":
                            big_lrf_value = txt
                        else:
                            small_lrf_value = txt
                    else:
                        buffer = buffer[1:]
            time.sleep(0.01)
    except Exception:
        # leave as "Error" if not connected / fails
        pass
    finally:
        try:
            ser.write(stop_cmd)
            ser.close()
        except Exception:
            pass

# ----------------------------
# Init models / devices
# ----------------------------
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")

# MiDaS
midas_weights = default_models[MIDAS_MODEL_TYPE]
midas_model, midas_transform, net_w, net_h = load_model(
    device,
    midas_weights,
    MIDAS_MODEL_TYPE,
    MIDAS_OPTIMIZE,
    height=MIDAS_HEIGHT,
    square=True,
)
if MIDAS_OPTIMIZE and device.type == "cuda":
    midas_model = midas_model.half()

torch.backends.cudnn.enabled = True
torch.backends.cudnn.benchmark = True

# YOLOs
detect_model = YOLO(DETECT_MODEL).to(0 if device.type == "cuda" else "cpu")
segment_model = YOLO(SEGMENT_MODEL).to(0 if device.type == "cuda" else "cpu")

# Start LRF threads (will just keep values at "Error" if ports not present)
threading.Thread(target=lrf_reader, args=(BIG_LRF_PORT,"BIG"), daemon=True).start()
threading.Thread(target=lrf_reader, args=(SMALL_LRF_PORT,"SMALL"), daemon=True).start()

# ----------------------------
# Video loop
# ----------------------------
cap = cv2.VideoCapture(VIDEO_PATH)
if not cap.isOpened():
    raise RuntimeError(f"Could not open video: {VIDEO_PATH}")

frame_w = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
frame_h = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
fps_in = cap.get(cv2.CAP_PROP_FPS) or 30

# optional writer
out_writer = None
if SAVE_OUTPUT:
    fourcc = cv2.VideoWriter_fourcc(*OUTPUT_CODEC)
    out_writer = cv2.VideoWriter(OUTPUT_PATH, fourcc, fps_in, (frame_w, frame_h))

win = "YOLO + Rails + Depth + LRF"
cv2.namedWindow(win, cv2.WND_PROP_FULLSCREEN)
cv2.setWindowProperty(win, cv2.WND_PROP_FULLSCREEN, cv2.WINDOW_FULLSCREEN)

prev_time = time.time()

while True:
    ret, frame = cap.read()
    if not ret:
        break

    annotated = frame.copy()

    # -------- Segmentation (rails ROI) --------
    rails_mask = np.zeros((frame_h, frame_w), dtype=np.uint8)
    seg_results = segment_model.predict(source=frame, device=0, conf=0.5, verbose=False)
    if seg_results and seg_results[0].masks is not None:
        center_x1 = int(frame_w * 0.4)
        center_x2 = int(frame_w * 0.6)
        best_overlap = 0.0
        best_polygon_mask = np.zeros((frame_h, frame_w), dtype=np.uint8)

        for polygon in seg_results[0].masks.xy:
            polygon = polygon.astype(int)
            poly_mask = np.zeros((frame_h, frame_w), dtype=np.uint8)
            cv2.fillPoly(poly_mask, [polygon], 1)
            overlap_ratio = np.mean(poly_mask[:, center_x1:center_x2])
            if overlap_ratio > best_overlap:
                best_overlap = overlap_ratio
                best_polygon_mask = poly_mask

        if best_overlap > 0.05:
            rows = frame_h
            for y in range(rows - 1, -1, -5):
                max_width = 500
                min_width = -500
                dilation_width = int(min_width + (max_width - min_width) * (y / rows))
                ksize = dilation_width if dilation_width % 2 == 1 else dilation_width + 1
                if ksize > 1:
                    kernel = cv2.getStructuringElement(cv2.MORPH_RECT, (ksize, 1))
                    slice_mask = best_polygon_mask[y:y+5, :]
                    dilated_slice = cv2.dilate(slice_mask, kernel, iterations=1)
                    rails_mask[y:y+5, :] = dilated_slice

    # blue overlay for rails
    if rails_mask.any():
        mask_3c = rails_mask[..., None]
        blue = np.zeros_like(annotated); blue[..., 0] = 255
        blended = cv2.addWeighted(blue, 0.5, annotated, 0.5, 0)
        annotated = np.where(mask_3c == 1, blended, annotated)

    # -------- Detection --------
    det_results = detect_model.predict(source=frame, device=0, conf=0.2, verbose=False)
    boxes = det_results[0].boxes.xyxy.cpu().numpy().astype(int) if det_results else []

    # -------- Depth (MiDaS on center 9:16 crop) --------
    cropped, xL, xR = crop_center_9_16(frame)  # x in [xL,xR)
    rgb = cv2.cvtColor(cropped, cv2.COLOR_BGR2RGB)
    image_in = midas_transform({"image": rgb / 255.0})["image"]
    with torch.no_grad():
        pred_raw = process_midas(
            device, midas_model, MIDAS_MODEL_TYPE,
            image_in, target_size=rgb.shape[1::-1], optimize=MIDAS_OPTIMIZE
        )

    Hc, Wc = pred_raw.shape
    ref_val = pred_raw[Hc - 1, Wc // 2]
    eps = 1e-8
    scale_k = ASSUME_BOTTOM_CENTER_M * max(ref_val, eps)  # z = k / d
    depth_meters = scale_k / (pred_raw + eps)

    # -------- Annotate detections with danger + depth --------
    for x1, y1, x2, y2 in boxes:
        x1, y1 = max(0, x1), max(0, y1)
        x2, y2 = min(frame_w - 1, x2), min(frame_h - 1, y2)

        box_area = max(1, (x2 - x1) * (y2 - y1))
        overlap = np.sum(rails_mask[y1:y2, x1:x2] > 0) / box_area
        danger = overlap > DANGER_OVERLAP_THRESH
        color = (0, 0, 255) if danger else (0, 255, 0)
        cv2.rectangle(annotated, (x1, y1), (x2, y2), color, 2)

        if danger:
            # intersect bbox with 9:16 crop, map into depth coords, show closest depth
            xc1 = max(x1, xL); xc2 = min(x2, xR)
            if xc2 > xc1:
                lx1 = max(0, min(Wc - 1, xc1 - xL))
                lx2 = max(0, min(Wc,     xc2 - xL))
                ly1 = max(0, min(Hc - 1, y1))
                ly2 = max(0, min(Hc,     y2))
                if lx2 > lx1 and ly2 > ly1:
                    patch = depth_meters[ly1:ly2, lx1:lx2]
                    if patch.size:
                        closest = float(np.min(patch))
                        cv2.putText(
                            annotated, f"{closest:.2f} m",
                            (x1, max(0, y1 - 8)),
                            cv2.FONT_HERSHEY_SIMPLEX, 0.6, color, 2, cv2.LINE_AA
                        )

    # -------- Overlay LRF values + FPS --------
    cv2.putText(annotated, f"BIG LRF: {big_lrf_value}", (10, frame_h-40),
                cv2.FONT_HERSHEY_SIMPLEX, 0.7, (255,255,0), 2)
    cv2.putText(annotated, f"SMALL LRF: {small_lrf_value}", (10, frame_h-10),
                cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0,255,255), 2)

    now = time.time()
    fps = 1.0 / max(now - prev_time, 1e-6)
    prev_time = now
    cv2.putText(annotated, f"FPS: {fps:.2f}", (10, 30),
                cv2.FONT_HERSHEY_SIMPLEX, 1, (0,255,0), 2)

    # show / save
    cv2.imshow(win, annotated)
    if SAVE_OUTPUT and out_writer is not None:
        out_writer.write(annotated)

    if cv2.waitKey(1) & 0xFF == ord('q'):
        break

cap.release()
if SAVE_OUTPUT and out_writer is not None:
    out_writer.release()
cv2.destroyAllWindows()


Device: cuda
initialize_weights...
Model loaded, number of parameters = 72M
MiDaS encoder input: 384x384
